In [8]:
import pandas as pd

df_booking  = pd.read_csv(r'C:\Users\KITTU-PALKIN\OneDrive\Documents\Rupahli\GUVI\Projects\Rapido\src\DataCSVFiles\bookings.csv')

df_booking = pd.get_dummies(df_booking,columns=['vehicle_type'],dtype='int')


from sklearn.preprocessing import OrdinalEncoder


encoder = OrdinalEncoder(categories=[[ "Low", #Ordinal
        "Medium",
        "High"]])

df_booking['traffic_level'] = encoder.fit_transform(df_booking[['traffic_level']])

encoder = OrdinalEncoder(categories=[[ "Clear", #Ordinal
        "Rain",
        "Heavy Rain"]])

df_booking['weather_condition'] = encoder.fit_transform(df_booking[['weather_condition']])

#df_booking['Fare_per_KM'] = df_booking['booking_value'] / df_booking['ride_distance_km']
#df_booking['Fare_per_Min'] = df_booking['booking_value'] / df_booking['estimated_ride_time_min']

#df_booking['Long_Distance_Flag'] = df_booking['ride_distance_km'] > 20
#df_booking['Long_Distance_Flag'] = df_booking['Long_Distance_Flag'].astype(int)

df_booking['booking_date'] = pd.to_datetime(df_booking['booking_date'])
#df_booking['booking_year'] = df_booking['booking_date'].dt.year
#df_booking['booking_month'] = df_booking['booking_date'].dt.month
#df_booking['booking_day'] = df_booking['booking_date'].dt.day


df_booking.drop(['is_weekend','base_fare','incomplete_ride_reason','city','estimated_ride_time_min','customer_id', 'driver_id','booking_date','booking_time','booking_id','actual_ride_time_min','pickup_location','drop_location','booking_status','day_of_week'], axis=1, inplace=True)

pd.set_option('display.max_columns', None)

df_booking.head()

,hour_of_day,ride_distance_km,traffic_level,weather_condition,surge_multiplier,booking_value,vehicle_type_Auto,vehicle_type_Bike,vehicle_type_Cab
0,0,7.01,2.0,2.0,2.0,148.22,0,1,0
1,6,9.67,1.0,2.0,1.8,465.85,0,0,1
2,8,16.18,0.0,2.0,1.9,457.03,1,0,0
3,10,1.02,1.0,1.0,1.8,51.03,0,1,0
4,0,12.35,1.0,0.0,1.2,144.73,0,1,0


In [9]:
import pandas as pd

df_loc  = pd.read_csv(r'C:\Users\KITTU-PALKIN\OneDrive\Documents\Rupahli\GUVI\Projects\Rapido\src\DataCSVFiles\location_demand.csv')

df_loc.head()

,city,pickup_location,hour_of_day,vehicle_type,total_requests,completed_rides,cancelled_rides,avg_wait_time_min,avg_surge_multiplier,demand_level
0,Bangalore,Loc_1,0,Auto,2,2,0,87.940000,1.400000,Low
1,Bangalore,Loc_1,0,Bike,5,5,0,68.088000,1.460000,Low
2,Bangalore,Loc_1,0,Cab,6,5,0,50.913333,1.733333,Medium
3,Bangalore,Loc_1,1,Auto,3,2,0,72.883333,1.566667,Low
4,Bangalore,Loc_1,1,Bike,7,6,1,33.374286,1.242857,Medium


In [10]:
import pandas as pd

df_time  = pd.read_csv(r'C:\Users\KITTU-PALKIN\OneDrive\Documents\Rupahli\GUVI\Projects\Rapido\src\DataCSVFiles\time_features.csv')

unique_time = df_time[['hour_of_day', 'peak_time_flag']].drop_duplicates(subset=['hour_of_day'])

In [11]:
df_booking = pd.merge(df_booking, unique_time[['hour_of_day','peak_time_flag']], on='hour_of_day', how='left')
df_booking.drop(['hour_of_day'], axis=1, inplace=True)

df_booking.head()
df_booking.shape

(100000, 9)

In [12]:
df_booking.head()

,ride_distance_km,traffic_level,weather_condition,surge_multiplier,booking_value,vehicle_type_Auto,vehicle_type_Bike,vehicle_type_Cab,peak_time_flag
0,7.01,2.0,2.0,2.0,148.22,0,1,0,0
1,9.67,1.0,2.0,1.8,465.85,0,0,1,0
2,16.18,0.0,2.0,1.9,457.03,1,0,0,1
3,1.02,1.0,1.0,1.8,51.03,0,1,0,1
4,12.35,1.0,0.0,1.2,144.73,0,1,0,0


In [13]:
X = df_booking.drop(['booking_value'],axis = 1)
Y = df_booking['booking_value']

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler


x_train,x_test , y_train,y_test = train_test_split(X,Y,test_size = 0.2,random_state = 42)


from sklearn.linear_model import Ridge,ElasticNet,Lasso,LinearRegression
from sklearn.pipeline import Pipeline

models = {"Linear Regression": LinearRegression(),
    "Lasso": Lasso(alpha=1.0),
    "Ridge": Ridge(alpha=1.0),
    "ElasticNet": ElasticNet(alpha=1.0, l1_ratio=0.5)}

for model_name, model in models.items():

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", model)
    ])

    pipeline.fit(x_train, y_train)


    train_prediction = pipeline.predict(x_train)

    test_prediction = pipeline.predict(x_test)

    from sklearn.metrics import r2_score,mean_squared_error

    print(f"{model_name} -- Train MSE :{mean_squared_error(y_train,train_prediction)}")
    print(f"{model_name} -- Train R2 :{r2_score(y_train,train_prediction)}")

    print(f"{model_name} -- Test MSE :{mean_squared_error(y_test,test_prediction)}")
    print(f"{model_name} -- Test R2 :{r2_score(y_test,test_prediction)}")

    print("\n")

    '''Linear Regression -- Train MSE :3720.7549588803768
Linear Regression -- Train R2 :0.9139984230588344
Linear Regression -- Test MSE :3685.9065055370957
Linear Regression -- Test R2 :0.9148638975397144


Lasso -- Train MSE :3724.375346630205
Lasso -- Train R2 :0.9139147413708263
Lasso -- Test MSE :3690.3872747950695
Lasso -- Test R2 :0.9147604019057164


Ridge -- Train MSE :3720.754963175788
Ridge -- Train R2 :0.9139984229595502
Ridge -- Test MSE :3685.9076756692425
Ridge -- Test R2 :0.9148638705123088'''

Linear Regression -- Train MSE :3720.8929051057844
Linear Regression -- Train R2 :0.9139952345680444
Linear Regression -- Test MSE :3685.638992478986
Linear Regression -- Test R2 :0.9148700764862207


Lasso -- Train MSE :3724.3753466302046
Lasso -- Train R2 :0.9139147413708263
Lasso -- Test MSE :3690.3872747950686
Lasso -- Test R2 :0.9147604019057165


Ridge -- Train MSE :3720.892909399382
Ridge -- Train R2 :0.9139952344688022
Ridge -- Test MSE :3685.640117068866
Ridge -- Test R2 :0.9148700505107383


ElasticNet -- Train MSE :7107.794891001454
ElasticNet -- Train R2 :0.8357103394456195
ElasticNet -- Test MSE :7102.647998773967
ElasticNet -- Test R2 :0.8359448979906099




In [ ]:
X = df_booking.drop(['booking_value'],axis = 1)
Y = df_booking['booking_value']

from sklearn.model_selection import train_test_split
import numpy as np



x_train,x_test , y_train,y_test = train_test_split(X,Y,test_size = 0.2,random_state = 42)


from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=42))
])

# 2. Define the grid of alpha values to test 
# Testing small fractions, whole numbers, and larger penalties
param_grid = {
    'ridge__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 200.0, 500.0]
}

# 3. Setup GridSearchCV
# cv=5 means it performs 5-fold cross-validation
# n_jobs=-1 uses all your computer's CPU cores to run fast
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error', 
    cv=5,
    n_jobs=-1
)

# 4. Fit the grid search on your training data
grid_search.fit(x_train, y_train)

# 5. Extract the results
best_alpha = grid_search.best_params_['ridge__alpha']
# Convert negative RMSE back to a positive value in Rupees
best_train_rmse = -grid_search.best_score_ 

print(f"🏆 Best Alpha value found: {best_alpha}")
print(f"📉 Optimized Train RMSE: {best_train_rmse:.2f} Rupees")



from sklearn.metrics import mean_squared_error

# Use the best estimator directly to predict on test data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(x_test)

final_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"🎯 Final Test RMSE: {final_test_rmse:.2f} Rupees")


tuned_ridge = grid_search.best_estimator_.named_steps['ridge']

# 2. Pair the original feature column names with their learned weights
coefficients_df = pd.DataFrame({
    'Feature Name': x_train.columns,
    'Weight (Standardized Impact)': tuned_ridge.coef_
})

# 3. Calculate the absolute impact to sort them by absolute importance
coefficients_df['Absolute Impact'] = coefficients_df['Weight (Standardized Impact)'].abs()
coefficients_df = coefficients_df.sort_values(by='Absolute Impact', ascending=False).drop(columns=['Absolute Impact'])

# 4. Display the results cleanly
print("--- Rapido Fare Feature Importance ---")
print(coefficients_df.to_string(index=False))

🏆 Best Alpha value found: 1.0
📉 Optimized Train RMSE: 61.01 Rupees
🎯 Final Test RMSE: 60.71 Rupees
--- Rapido Fare Feature Importance ---
     Feature Name  Weight (Standardized Impact)
 ride_distance_km                    139.197655
 vehicle_type_Cab                     76.925074
vehicle_type_Bike                    -65.600129
 surge_multiplier                     34.170519
weather_condition                     26.278905
    traffic_level                     16.788419
   peak_time_flag                     14.121769
vehicle_type_Auto                    -11.383032


: 